# KuchoLM training

日本語コーパスを NIDA_FICTION へ変換し、小型 seq2seq Transformer を学習する notebook です。

データ生成済みの場合は `/content/kucholm_nida.jsonl` を使って tokenizer 学習 → モデル学習 → 保存 → 推論まで実行できます。

In [ ]:
!pip -q install sentencepiece torch

## 1. 設定

In [ ]:
from pathlib import Path
import json, math, random
import sentencepiece as spm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

DATA_PATH = Path('/content/kucholm_nida.jsonl')
WORK_DIR = Path('/content/kucholm_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 2. 学習データ読み込み

In [ ]:
rows = []
with DATA_PATH.open(encoding='utf-8') as f:
    for line in f:
        x = json.loads(line)
        rows.append((f"<NIDA_FICTION> {x['source']}", x['target']))
random.shuffle(rows)
cut = max(1, int(len(rows) * 0.98))
train_rows = rows[:cut]
val_rows = rows[cut:]
print('train:', len(train_rows), 'val:', len(val_rows))

## 3. SentencePiece

In [ ]:
spm_input = WORK_DIR / 'spm_train.txt'
with spm_input.open('w', encoding='utf-8') as f:
    for src, tgt in train_rows:
        f.write(src.replace('\n', ' ') + '\n')
        f.write(tgt.replace('\n', ' ') + '\n')

spm.SentencePieceTrainer.train(
    input=str(spm_input),
    model_prefix=str(WORK_DIR / 'kucholm_spm'),
    vocab_size=8000,
    model_type='bpe',
    character_coverage=0.9995,
    pad_id=0, unk_id=1, bos_id=2, eos_id=3,
    user_defined_symbols=['<NIDA_FICTION>'],
)
sp = spm.SentencePieceProcessor(model_file=str(WORK_DIR / 'kucholm_spm.model'))
PAD, UNK, BOS, EOS = 0, 1, 2, 3
VOCAB = sp.vocab_size()
print('vocab:', VOCAB)

## 4. Dataset / DataLoader

In [ ]:
MAX_LEN = 128
BATCH = 32 if device.type == 'cuda' else 8

def encode(text):
    ids = [BOS] + sp.encode(text, out_type=int)[:MAX_LEN-2] + [EOS]
    return ids

class PairDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        src, tgt = self.data[i]
        return torch.tensor(encode(src)), torch.tensor(encode(tgt))

def collate(batch):
    srcs, tgts = zip(*batch)
    src = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD)
    tgt = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD)
    return src, tgt

train_loader = DataLoader(PairDataset(train_rows), batch_size=BATCH, shuffle=True, collate_fn=collate)
val_loader = DataLoader(PairDataset(val_rows), batch_size=BATCH, shuffle=False, collate_fn=collate)

## 5. KuchoLM NIDA-15M

In [ ]:
D_MODEL = 320
NHEAD = 8
ENC_LAYERS = 4
DEC_LAYERS = 4
FF = 1280
DROPOUT = 0.1
EPOCHS = 5
LR = 2.5e-4

class KuchoTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.pos = nn.Embedding(MAX_LEN, D_MODEL)
        self.tf = nn.Transformer(
            d_model=D_MODEL, nhead=NHEAD,
            num_encoder_layers=ENC_LAYERS,
            num_decoder_layers=DEC_LAYERS,
            dim_feedforward=FF, dropout=DROPOUT,
            batch_first=True, norm_first=True
        )
        self.lm_head = nn.Linear(D_MODEL, VOCAB, bias=False)
        self.lm_head.weight = self.embed.weight

    def add_pos(self, x):
        p = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        return self.embed(x) * math.sqrt(D_MODEL) + self.pos(p)

    def forward(self, src, tgt_in):
        src_pad = src.eq(PAD)
        tgt_pad = tgt_in.eq(PAD)
        mask = nn.Transformer.generate_square_subsequent_mask(tgt_in.size(1), device=tgt_in.device)
        h = self.tf(
            self.add_pos(src), self.add_pos(tgt_in),
            tgt_mask=mask,
            src_key_padding_mask=src_pad,
            tgt_key_padding_mask=tgt_pad,
            memory_key_padding_mask=src_pad,
        )
        return self.lm_head(h)

model = KuchoTransformer().to(device)
params = sum(p.numel() for p in model.parameters())
print(f'{params/1e6:.2f}M parameters')

## 6. 学習

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.98), weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.05)
scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
best_val = float('inf')
best_path = WORK_DIR / 'KuchoLM-NIDA-15M.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=device.type == 'cuda'):
            logits = model(src, tgt_in)
            loss = criterion(logits.reshape(-1, VOCAB), tgt_out.reshape(-1))
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)
            logits = model(src, tgt[:, :-1])
            loss = criterion(logits.reshape(-1, VOCAB), tgt[:, 1:].reshape(-1))
            val_loss += loss.item()

    train_loss /= max(1, len(train_loader))
    val_loss /= max(1, len(val_loader))
    print(f'epoch {epoch}: train={train_loss:.4f} val={val_loss:.4f}')
    if val_loss < best_val:
        best_val = val_loss
        torch.save({
            'model': model.state_dict(),
            'config': {
                'vocab': VOCAB, 'd_model': D_MODEL, 'nhead': NHEAD,
                'enc_layers': ENC_LAYERS, 'dec_layers': DEC_LAYERS,
                'ff': FF, 'max_len': MAX_LEN
            },
            'best_val': best_val,
        }, best_path)
        print('saved best:', best_path)

## 7. 推論

In [ ]:
checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint['model'])
model.eval()

@torch.no_grad()
def infer(text, max_new_tokens=96, repetition_penalty=1.15):
    src_ids = encode('<NIDA_FICTION> ' + text)
    src = torch.tensor([src_ids], device=device)
    out = [BOS]
    max_steps = min(max_new_tokens, max(12, len(src_ids) + 24), MAX_LEN - 1)
    for _ in range(max_steps):
        tgt = torch.tensor([out], device=device)
        logits = model(src, tgt)[0, -1].clone()
        for token_id in set(out[1:]):
            if logits[token_id] > 0:
                logits[token_id] /= repetition_penalty
            else:
                logits[token_id] *= repetition_penalty
        next_id = int(torch.argmax(logits))
        if next_id == EOS:
            break
        out.append(next_id)
    return sp.decode(out[1:])

for s in [
    '今日は学校です。',
    '明日は雨が降るかもしれません。',
    '最近少し暖かくなってきました。',
]:
    print(s, '->', infer(s))

print('saved:', best_path)